# 1. Getting started - A single layer

This notebook demonstrates how to instantiate and configure a single layer, emulate, get resource and latency estimates, and finally how to simulate and compile with Quartus. Everything in this tutorial can be applied to any future notebooks and graphs.


In [1]:
from pathlib import Path

import numpy as np
from fpga_ai_suite.spatial.flow import run_flow # Utility to simulate and compile with Quartus.
from fpga_ai_suite.spatial.layers import Dot

WORK = Path("work_getting_started")
WORK.mkdir(exist_ok=True)


## Creating a Dot Layer

Arguments:
- weights: The feature_in x feature_out weight matrix as an fp32 or integer numpy array. If 2D, the same weights get applied to the same inputs. If 3D the 2D weights get rotated round robin.
- num_packets_per_weight: The number of inputs required to trigger a rotation if the weights are 3D. Default 1
- num_lanes_in: The number of input elements processed in parallel per clock cycle. Default is config.num_lanes
- num_lanes_out: The number of elements that are produced in parallel per clock cycle. Default is config.num_lanes
- dtype: The data type of the layer. "INT" or "BF16"
- weight_exponent_size: The number of bits in the exponent in BF16 mode. Must be < 8.
- weight_mantissa_size: The number of bits in the mantissa in BF16 mode. Must be < 7.
- mif_init: Initialize all weights on chip or source from a channel during runtime. (Used to enable DDR if false)
- device_family: The target Altera device family. "Agilex 7" or "Agilex 5".


In [2]:
# Weight matrix: features_in x features_out. Both dims must divide by lane counts.
weights = np.random.randn(64, 128).astype(np.float32)
model = Dot(weights, num_lanes_in=4, num_lanes_out=8, dtype="BF16", mif_init=True)

## Shapes: `(P, N, M)`

Every streaming tensor is a 3-D array:

| Axis | Meaning |
|------|---------|
| **P** | Packet count — temporal dimension; packets arrive one after another |
| **N** | Rows in the spatial vector |
| **M** | Columns in the spatial vector |

So `(1, 1, M)` is a single row vector with `M` elements.

Layer shape rules differ. Read the **`forward` docstring** on each layer:

- **Dot** expects `(P, 1, M)` (row vectors) — see `Dot.forward`
- **Transpose** / **Matmul** use full `(P, N, M)` tensors

Also: `M % num_lanes == 0` must be true for every port.


In [3]:
# Dot needs (P, 1, features_in). features_in == weights.shape[-2] == 64.
P = 2
data = np.random.randn(P, 1, 64).astype(np.float32)
print("input shape (P, N, M) =", data.shape)


input shape (P, N, M) = (2, 1, 64)


## Emulation (software forward)

Two equivalent ways to run the bit-accurate emulator for basic_layers:


In [4]:
# 1) Pass inputs into forward call
out = model(data)
print("via arguments:", out.shape)

# 2) Set IO_port.data yourself, then call forward with no arguments
model.inputs.din.data = data
out_manual = model()
print("via .data + forward():", out_manual.shape)
np.testing.assert_array_equal(out, out_manual)

via arguments: (2, 1, 128)
via .data + forward(): (2, 1, 128)


## `set_shape` when you do not need data

Spatial IP is shaped by tensor dimensions: hardware widths and depths come from
those shapes. Shapes must come from either `forward` (which sets them from data)
or `set_shape` (shape only). This function allows you to go through many flows (i.e., quartus_compile/save_model) without having to run an expensive forward call.

Warning: You will need to create a new object if you want to change the shape after it has already been set to something different.


In [5]:
model.set_shape((P, 1, 64))
print("din shape:", model.inputs.din.shape)
print("dout shape:", model.outputs.dout.shape)

# 2) Set IO_port.shape yourself, then call set_shape with no arguments
model.inputs.din.shape = (P, 1, 64)
model.set_shape()
print("din shape (manual):", model.inputs.din.shape)
print("dout shape (manual):", model.outputs.dout.shape)

din shape: (2, 1, 64)
dout shape: (2, 1, 128)
din shape (manual): (2, 1, 64)
dout shape (manual): (2, 1, 128)


## Latency and resource estimates

Call these **only after** `set_shape` or `forward`.

The data_timing attribute must be set with a list of tuples (start of packet, end of packet) to estimate timing.


In [6]:
num_lanes_in = model.inputs.din.num_lanes
input_packet_size = np.prod(model.inputs.din.shape[-2:]) // num_lanes_in
# Two packets back to back
input_timing = [(0, input_packet_size - 1), (input_packet_size, input_packet_size*2 - 1)]
model.inputs.din.data_timing = input_timing
output_timing = model.estimate_timing()
print("Input timing (cycles): ", input_timing)
print("Output timing (cycles):", output_timing)
print("Latency in cycles (first input SOP to first output SOP)", output_timing[0][0] - input_timing[0][0])
print("resources:", model.estimate_resource_utilization())


Input timing (cycles):  [(0, 15), (16, 31)]
Output timing (cycles): [(257, 272), (513, 528)]
Latency in cycles (first input SOP to first output SOP) 257
resources: {'ALM': 988, 'DSP': 17, 'M20K': 16, 'M80K': 0}


## `run_flow` — Simulation and Quartus Compile tool

Call `set_shape` or `forward` before `run_flow`. For **simulation**, call
`forward` first so `dump_files/{port_name}.txt` holds the hex vectors from
emulation (that is what the RTL testbench checks against).

Modes are mutually exclusive: `sim=True` and `quartus_compile=True` must be separate calls.


## `run_flow` — RTL simulation

Optional kwargs (change in this same call if you need them):

- `simulation_tool="vcs"` (default) or `"questa"`
- `dump_waveform=True` — emit waveforms (VPD/WLF depending on tool)

Pass `gcc_path` pointing at GCC <= 13.2.0 for VCS if the default environment path is newer.


In [7]:
ret = run_flow(
    model,
    sim=True,
    recompile=True,
    # gcc_path=<path>,
    # simulation_tool="questa",  # optional alternative to default "vcs"
    # dump_waveform=True,        # optional waveforms
)


Dumping input and output data...
Dumping weights: matrix size: [64,128] total weights: 4096
Running model DOT_BF16_64_128_0_4_8_1_1_1_8_7_2_Agilex_7_32

Running Simulation (vcs)...
Running VCS in sim_DOT_BF16_64_128_0_4_8_1_1_1_8_7_2_Agilex_7_32 (device_family=Agilex 7)
*** Using c compiler gcc instead of cc ...
                         Chronologic VCS (TM)
      Version V-2023.12-SP2-1_Full64 -- Fri Sep  4 12:54:29 2026

                    Copyright (c) 1991 - 2024 Synopsys, Inc.
   This software and the associated documentation are proprietary to Synopsys,
 Inc. This software may only be used in accordance with the terms and conditions
 of a written license agreement with Synopsys, Inc. All other use, reproduction,
   or distribution of this software is strictly prohibited.  Licensed Products
     communicate with Synopsys servers for the purpose of providing software
    updates, detecting software piracy and verifying that customers are using
    Licensed Products in conformity wi

/nfs/site/disks/psg_ctools_1/gcc/13.2.0/1/linux64/rhel7/bin/../lib/gcc/x86_64-none-linux/13.2.0/../../../../x86_64-none-linux/bin/ld: warning: rmar_nd.o: missing .note.GNU-stack section implies executable stack
/nfs/site/disks/psg_ctools_1/gcc/13.2.0/1/linux64/rhel7/bin/../lib/gcc/x86_64-none-linux/13.2.0/../../../../x86_64-none-linux/bin/ld: NOTE: This behaviour is deprecated and will be removed in a future version of the linker


rm -f _cuarc*.so _csrc*.so pre_vcsobj_*.so share_vcsobj_*.so
if [ -x ../simv ]; then chmod a-x ../simv; fi
g++  -o ../simv   -no-pie    -rdynamic  -Wl,-rpath='$ORIGIN'/simv.daidir -Wl,-rpath=./simv.daidir -Wl,-rpath=/p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib -L/p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib  -Wl,-rpath-link=./   objs/amcQw_d.o   _17101_archive_1.so   SIM_l.o       rmapats_mop.o rmapats.o rmar.o rmar_nd.o  rmar_llvm_0_1.o rmar_llvm_0_0.o            -lnuma -lvirsim -lerrorinf -lsnpsmalloc -lvfs    -lvcsnew -lsimprofile -lreader_common /p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib/libBA.a -luclinative /p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib/vcs_tls.o   -Wl,-whole-archive  -lvcsucli    -Wl,-no-whole-archive       ./../simv.daidir/vc_hdrs.o    /p/psg/eda/synopsys/vcsmx/V-2023.12-SP2-1/linux64/suse/suse64/lib/vcs_save_restore_new.o -ldl  -lc -lm -lpthread -ldl 
/nfs/site/disks/psg_ctools_1

## `run_flow` — Quartus compile

Depending on the device family provided, run_flow will select either A5ED065BB32AE6S for Agilex 5, or AGIB027R29A1E1VB (Agilex 7) otherwise.

In [8]:
ret = run_flow(model, quartus_compile=True, seed=1)

Dumping input and output data...
Dumping weights: matrix size: [64,128] total weights: 4096
Running model DOT_BF16_WRAPPER

Running Quartus compile...
Info: *******************************************************************
Info: Running Quartus Prime Shell
    Info: Version 26.1.0 Build 110 03/26/2026 SC Pro Edition
    Info: Copyright (C) 2026  Altera Corporation. All rights reserved.
    Info: Your use of Altera Corporation's design tools, logic functions 
    Info: and other software and tools, and any partner logic 
    Info: functions, and any output files from any of the foregoing 
    Info: (including device programming or simulation files), and any 
    Info: associated documentation or information are expressly subject 
    Info: to the terms and conditions of the Altera Program License 
    Info: Subscription Agreement, the Altera Quartus Prime License Agreement,
    Info: the Altera IP License Agreement, or other applicable license
    Info: agreement, including, without l

## Config

The cells below cover the `Config` structure to make configuring layers easier.

### Global defaults via `Config`

Many layer defaults (`num_lanes`, `dw`, `dtype`, `mif_init`, `device_family`, …)
come from [`config.py`](../../../python/spark/src/fpga_ai_suite/spatial/layers/config.py).

You can edit that file to apply global changes, or mutate the singleton **before any graph or layer is
constructed** for local changes. Once any object reads config, further updates are rejected.


In [9]:
# Illustrative only — do this at process start, before building models:
#
from fpga_ai_suite.spatial.layers.config import Config
# Config.num_lanes = 8
# Config.dw = 16
# Config.dtype = "BF16"
# Config.device_family = "Agilex 7"
#
# model = Dot(...)   # first construction after Config updates

print("current defaults:",
      "num_lanes=", Config.get_config("num_lanes"),
      "dw=", Config.get_config("dw"),
      "dtype=", Config.get_config("dtype"),
      "device_family=", Config.get_config("device_family"))


current defaults: num_lanes= 4 dw= 16 dtype= BF16 device_family= Agilex 7
